# MS-Spectral-Foundation: Happy Path Tutorial

This notebook demonstrates the **full pipeline** by calling the two main scripts directly:

1. **`train.py`** — runs the complete data-loading + self-supervised training pipeline  
   (MGF → Module 1 parse → Module 2 filter → Module 3 bin → Module 4-6 SSL training)  

2. **`run_embedding_analysis.py`** — loads the trained model and runs embedding extraction + downstream analysis  
   (Module 6 embeddings → Module 7 distribution analysis → Module 8 exemplar discovery)

**To run on a different dataset:** change the MGF paths and model path in the configuration cells below.


## Section 1: Setup — Paths & sys.path

In [1]:
# ============================================================
# Dependency setup — safe for ALL environments (Anaconda, venv, plain Python)
#
# Strategy: check each package BEFORE trying to install it.
# Only missing packages are installed → already-installed C-extensions
# (numpy, pyarrow, etc.) are never upgraded while loaded → no kernel crash.
# ============================================================
import importlib.util, subprocess, sys

# Map: importable module name → pip install name
_packages = {
    "lightning"        : "lightning",
    "pytorch_lightning": "pytorch-lightning",
    "selfies"          : "selfies>=2.1.1",
    "pyteomics"        : "pyteomics>=4.7.2",
    "rdkit"            : "rdkit",
    "spectrum_utils"   : "spectrum-utils>=0.4.1",
    "PIL"              : "pillow>=9.4.0",
    "einops"           : "einops>=0.4.1",
    "lance"            : "lance",
    "polars"           : "polars>=0.19.0",
    "pyarrow"          : "pyarrow>=12.0.1",
    "lark"             : "lark>=1.1.4",
    "sortedcontainers" : "sortedcontainers>=2.4.0",
    "dill"             : "dill>=0.3.6",
    "psims"            : "psims>=1.3.3",
    "cloudpathlib"     : "cloudpathlib>=0.18.1",
    "appdirs"          : "appdirs",
    "click"            : "click",
    "natsort"          : "natsort",
    "psutil"           : "psutil",
    "yaml"             : "PyYAML",
    "requests"         : "requests",
    "rich"             : "rich-click>=1.6.1",
    "tensorboard"      : "tensorboard",
    "tqdm"             : "tqdm",
    "sklearn"          : "scikit-learn",
    "pandas"           : "pandas",
    "matplotlib"       : "matplotlib",
    "seaborn"          : "seaborn",
}

_to_install = [
    pip_name
    for mod, pip_name in _packages.items()
    if importlib.util.find_spec(mod) is None
]

if not _to_install:
    print("✓ All packages already available — nothing to install.")
else:
    print(f"Installing {len(_to_install)} missing package(s)...")
    _failed = []
    for pkg in _to_install:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", pkg],
            capture_output=True,
        )
        if r.returncode == 0:
            print(f"  ✓ {pkg}")
        else:
            _failed.append(pkg)
            print(f"  ✗ {pkg}: {r.stderr.decode().strip()[:150]}")

    if _failed:
        print(f"\n⚠  Could not install: {_failed}")
        print("   Try manually:  pip install <package>  or  conda install -c conda-forge <package>")
    else:
        print("\n✓ All missing packages installed. Restarting kernel...")
        import IPython
        IPython.Application.instance().kernel.do_shutdown(restart=True)


✓ All packages already available — nothing to install.


In [2]:
import sys
import os
import types
from pathlib import Path

# ---------------------------------------------------------------------------
# Resolve project root (MS-Spectral-Foundation/) regardless of where the
# notebook is opened from.
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path(".").resolve()           # tutorials/
PROJECT_ROOT = NOTEBOOK_DIR.parent           # MS-Spectral-Foundation/

for p in [
    str(PROJECT_ROOT),
    str(PROJECT_ROOT / "casanovo"),
    str(PROJECT_ROOT / "depthcharge"),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ---------------------------------------------------------------------------
# Stub out packages with no Python 3.13 wheels that depthcharge imports at
# the top level but which are never called at runtime with MGF files.
# Must be injected BEFORE depthcharge is first imported.
# ---------------------------------------------------------------------------

# --- timsrust_pyo3: Bruker timsTOF .d support ---
if "timsrust_pyo3" not in sys.modules:
    _ts = types.ModuleType("timsrust_pyo3")
    class _SpectrumReaderStub:
        def __init__(self, *a, **kw):
            raise RuntimeError("timsrust_pyo3 not installed — .d files unsupported; use MGF/mzML.")
    _ts.SpectrumReader = _SpectrumReaderStub
    sys.modules["timsrust_pyo3"] = _ts
    print("  [stub] timsrust_pyo3 (Bruker .d support disabled)")

# --- lance / lance.torch.data: LanceDataset used by depthcharge's own
#     SpectrumDataset; we use MS_Spectral_Foundation's SpectrumDataset instead
#     so LanceDataset is never actually called at runtime. ---
try:
    import lance  # noqa: F401  succeeds if installed
except ImportError:
    _lance         = types.ModuleType("lance")
    _lance_torch   = types.ModuleType("lance.torch")
    _lance_td      = types.ModuleType("lance.torch.data")

    class _LanceDatasetStub:
        def __init__(self, *a, **kw):
            raise RuntimeError("lance not installed — LanceDataset unavailable; use SpectrumDataset instead.")

    _lance_td.LanceDataset = _LanceDatasetStub
    _lance.torch = _lance_torch
    _lance_torch.data = _lance_td
    sys.modules["lance"]            = _lance
    sys.modules["lance.torch"]      = _lance_torch
    sys.modules["lance.torch.data"] = _lance_td
    print("  [stub] lance / lance.torch.data (LanceDataset disabled — not needed for MGF)")

print("Project root :", PROJECT_ROOT)
print("sys.path[0]  :", sys.path[0])


  [stub] timsrust_pyo3 (Bruker .d support disabled)
Project root : C:\Users\Lenovo\Desktop\576dataset\Github_project\MS-Spectral-Foundation
sys.path[0]  : C:\Users\Lenovo\Desktop\576dataset\Github_project\MS-Spectral-Foundation\depthcharge


## Section 2: Configuration

Edit the paths below to point to your own `.mgf` files. Everything else uses the same defaults as `train.py` and `run_embedding_analysis.py`.

In [3]:
# ---------------------------------------------------------------------------
# DATA PATHS — change these to your own MGF files
# ---------------------------------------------------------------------------
# For train.py: a single MGF is used for both training and validation
TRAIN_MGF = str(PROJECT_ROOT / "casanovo" / "sample_data" / "sample_preprocessed_spectra.mgf")
VAL_MGF   = TRAIN_MGF     # use the same file for validation in this demo

# For run_embedding_analysis.py: can supply multiple MGF files (one per group)
ANALYSIS_MGF_PATHS = [TRAIN_MGF]        # add more paths for multi-group analysis
MANUAL_GROUP_LABELS = ["Sample"]        # one label per path; or None to auto-detect

# ---------------------------------------------------------------------------
# TRAINED MODEL PATH — will be set automatically after training
# ---------------------------------------------------------------------------
# Override this if you already have a checkpoint saved elsewhere:
MODEL_PATH_OVERRIDE = None   # e.g. r"C:\...\model_ssl_v4_epoch019.pt"

# Output directories
TRAIN_OUTPUT_DIR    = str(PROJECT_ROOT / "outputs")
ANALYSIS_OUTPUT_DIR = str(PROJECT_ROOT / "logs" / "embedding_analysis")

os.makedirs(TRAIN_OUTPUT_DIR,    exist_ok=True)
os.makedirs(ANALYSIS_OUTPUT_DIR, exist_ok=True)

print("Train MGF   :", TRAIN_MGF)
print("Train output:", TRAIN_OUTPUT_DIR)
print("Analysis out:", ANALYSIS_OUTPUT_DIR)


Train MGF   : C:\Users\Lenovo\Desktop\576dataset\Github_project\MS-Spectral-Foundation\casanovo\sample_data\sample_preprocessed_spectra.mgf
Train output: C:\Users\Lenovo\Desktop\576dataset\Github_project\MS-Spectral-Foundation\outputs
Analysis out: C:\Users\Lenovo\Desktop\576dataset\Github_project\MS-Spectral-Foundation\logs\embedding_analysis


## Section 3: Run `train.py` — Self-Supervised Training (Modules 1–6)

This calls `train.main()` directly. Internally it executes:

| Module | File | What it does |
|--------|------|-------------|
| 1 | `mgf_parse.py` | Parse spectra from the MGF file |
| 2 | `peak_filter.py` | Filter and normalise peaks |
| 3 | `bin_mz.py` | Pre-compute integer bin labels |
| 4 | `model_ssl.py` `mask_spectrum()` | BERT-style masking |
| 5 | `model_ssl.py` `forward()` + `training_step()` | SSL transformer training |
| 6 | `model_ssl.py` `get_embeddings()` | Save model checkpoint |


In [4]:
import glob
import MS_Spectral_Foundation.train as train_module

# ---------------------------------------------------------------------------
# Check for existing checkpoint; if found skip training
# ---------------------------------------------------------------------------
_DEFAULT_PT = str(PROJECT_ROOT / "outputs" / "model_ssl_v4_epoch019.pt")
TRAINED_MODEL_PATH = MODEL_PATH_OVERRIDE  # honour explicit override if set

if TRAINED_MODEL_PATH is None:
    if Path(_DEFAULT_PT).exists():
        TRAINED_MODEL_PATH = _DEFAULT_PT
        print(f"[Skip training] Found existing checkpoint:\n  {TRAINED_MODEL_PATH}")
    else:
        print("Starting training via train.main() ...\n")

        # Directly call train.main() with notebook-configured paths
        train_module.main(config_override={
            "train_mgf"   : TRAIN_MGF,
            "val_mgf"     : VAL_MGF,
            "batch_size"  : 4,
            "lr"          : 1e-4,
            "max_epochs"  : 3,
            "num_workers" : 0,
            "output_dir"  : TRAIN_OUTPUT_DIR,
        })

        pt_files = sorted(glob.glob(os.path.join(TRAIN_OUTPUT_DIR, "*.pt")))
        if pt_files:
            TRAINED_MODEL_PATH = pt_files[-1]
        else:
            ckpt_files = sorted(glob.glob(
                os.path.join(TRAIN_OUTPUT_DIR, "lightning_logs", "**", "*.ckpt"),
                recursive=True,
            ))
            TRAINED_MODEL_PATH = ckpt_files[-1] if ckpt_files else None

        print(f"\nTraining complete. Checkpoint → {TRAINED_MODEL_PATH}")

print(f"\nTRAINED_MODEL_PATH = {TRAINED_MODEL_PATH}")


c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Skip training] Found existing checkpoint:
  C:\Users\Lenovo\Desktop\576dataset\Github_project\MS-Spectral-Foundation\outputs\model_ssl_v4_epoch019.pt

TRAINED_MODEL_PATH = C:\Users\Lenovo\Desktop\576dataset\Github_project\MS-Spectral-Foundation\outputs\model_ssl_v4_epoch019.pt


## Section 4: Run `run_embedding_analysis.py` — Embedding Analysis (Modules 7–8)

This calls `run_embedding_analysis.main()` with a patched config to use the model trained above.

| Module | What it does |
|--------|-------------|
| 7.1 | Sample-level aggregation of spectrum embeddings |
| 7.2 | Embedding-space distribution analysis |
| 7.3 | Exemplar spectrum discovery |
| 8 | Visualisation & non-supervised evaluation |


In [5]:
import MS_Spectral_Foundation.run_embedding_analysis as emb_module

# ---------------------------------------------------------------------------
# Validate model path before running
# ---------------------------------------------------------------------------
_model_path = MODEL_PATH_OVERRIDE or TRAINED_MODEL_PATH
if _model_path is None or not Path(_model_path).exists():
    raise FileNotFoundError(
        f"No trained model found at: {_model_path}\n"
        "Please run Section 3 (training) first, or set MODEL_PATH_OVERRIDE."
    )

print(f"Using model: {_model_path}\n")

# ---------------------------------------------------------------------------
# Directly call run_embedding_analysis.main()
# Internally it calls _cache_key(), _load_cache(), _save_cache() from the module
# ---------------------------------------------------------------------------
emb_module.main(config_override={
    "model_path"          : _model_path,
    "mgf_paths"           : ANALYSIS_MGF_PATHS,
    "manual_group_labels" : MANUAL_GROUP_LABELS,
    "max_peaks"           : 150,
    "mz_min"              : 50.0,
    "mz_max"              : 2500.0,
    "bin_size"            : 0.5,
    "batch_size"          : 4,
    "num_workers"         : 0,
    "embedding_type"      : "cls",
    "n_exemplars"         : 5,
    "output_dir"          : ANALYSIS_OUTPUT_DIR,
    "cache_dir"           : str(PROJECT_ROOT / "logs" / "embedding_cache"),
})


Using model: C:\Users\Lenovo\Desktop\576dataset\Github_project\MS-Spectral-Foundation\outputs\model_ssl_v4_epoch019.pt

MS-Spectral-Foundation: Embedding Analysis Pipeline
Modules 7 & 8: Distribution Analysis and Representation Evaluation
[Cache] Cache hit (id=1adabfaa), loading directly and skipping Modules 1-6
  Shape: (128, 512)
  Metadata rows: 128
[Init] n_bins automatically set to 4900
   Model loaded from C:\Users\Lenovo\Desktop\576dataset\Github_project\MS-Spectral-Foundation\outputs\model_ssl_v4_epoch019.pt
   Device: cpu

Preparing Sample Metadata
   Prepared analysis data:
   Embeddings: (128, 512)
   Samples: 1 unique
   Groups: {0: 'Sample'}

  Sample metadata summary:
    Total spectra: 128
    Unique samples: 1
    Sample: 128 spectra

   Inter-group distance analysis requires at least 2 groups.
   Please provide MGF files from different disease groups.

   Current options:
   1. Add HCC data: Uncomment the HCC MGF path in config
   2. Use a combined MGF file with TITLE 

## Summary

| Step | Script called | Modules covered |
|------|--------------|----------------|
| Training | `train.py` → `main()` | 1 (parse) · 2 (filter) · 3 (bin) · 4–6 (SSL model) |
| Analysis | `run_embedding_analysis.py` → `main()` | 6 (embeddings) · 7 (distribution) · 8 (evaluation) |

**Outputs written to:**
- Trained model checkpoint → `outputs/`  
- Embedding analysis plots and reports → `logs/embedding_analysis/`

**To run on a different dataset:** update `TRAIN_MGF`, `VAL_MGF`, `ANALYSIS_MGF_PATHS`, and `MANUAL_GROUP_LABELS` in Section 2, then re-execute Sections 3 and 4.
